In [3]:
from ultralytics import YOLO
import math
import csv
import os
from datetime import datetime
import re
import glob

def calculate_angle(p1, p2, p3):
    """
    Calculates the angle between three points, used for joint curvature analysis.
    p1, p2, p3 = [x, y]
    """
    angle = math.degrees(math.atan2(p3[1] - p2[1], p3[0] - p2[0]) - 
                         math.atan2(p1[1] - p2[1], p1[0] - p2[0]))
    if angle < 0:
        angle += 360
    return angle

def analyze_posture(front_image_path, side_image_path, model_path='../../models/pose/yolov8s-pose.pt'):
    """
    Analyzes posture from a front and side image.
    - Front image is used for shoulder asymmetry.
    - Side image is used for forward head posture (slouch angle).
    Returns a dictionary containing shoulder_asymmetry and slouch_angle.
    """
    try:
        model = YOLO(model_path)
        shoulder_asymmetry = None
        slouch_angle = None

        # --- Part 1: Analyze Front Image for Shoulder Asymmetry ---
        print(f"--- Analyzing Front View: {front_image_path} ---")
        front_results = model(front_image_path, verbose=False)
        front_keypoints = front_results[0].keypoints.xy[0].numpy()
        
        if len(front_keypoints) > 0 and front_keypoints.shape[0] >= 7:
            left_shoulder = front_keypoints[5]
            right_shoulder = front_keypoints[6]
            shoulder_asymmetry = abs(left_shoulder[1] - right_shoulder[1])
            print(f"Shoulder Asymmetry (Y-diff): {shoulder_asymmetry:.2f} pixels.")
            # front_results[0].save("posture_front_result.jpg")
        else:
            print("Warning: No person or insufficient keypoints detected in the front image.")

        # --- Part 2: Analyze Side Image for Slouch Angle ---
        print(f"\n--- Analyzing Side View: {side_image_path} ---")
        side_results = model(side_image_path, verbose=False)
        side_keypoints = side_results[0].keypoints.xy[0].numpy()

        if len(side_keypoints) > 0 and side_keypoints.shape[0] >= 12:
            # We need to determine which side is visible (left or right)
            # We check for the presence of ear, shoulder, and hip on one side.
            # Let's assume the left side is more likely to be visible.
            ear = side_keypoints[3]      # left_ear
            shoulder = side_keypoints[5] # left_shoulder
            hip = side_keypoints[11]     # left_hip

            # If left ear is not detected, try right side
            if ear[0] == 0 and ear[1] == 0:
                print("Left ear not detected, trying right side keypoints for slouch angle.")
                ear = side_keypoints[4]      # right_ear
                shoulder = side_keypoints[6] # right_shoulder
                hip = side_keypoints[12]     # right_hip

            slouch_angle = calculate_angle(ear, shoulder, hip)
            if slouch_angle > 180:
                slouch_angle = 360 - slouch_angle
            
            print(f"Slouch Angle (Ear-Shoulder-Hip): {slouch_angle:.2f} degrees.")
            # side_results[0].save("posture_side_result.jpg")
        else:
            print("Warning: No person or insufficient keypoints detected in the side image.")

        if shoulder_asymmetry is not None and slouch_angle is not None:
            return {
                "shoulder_asymmetry": shoulder_asymmetry,
                "slouch_angle": slouch_angle
            }
        else:
            return None

    except Exception as e:
        print(f"An error occurred during posture analysis: {e}")
        return None

def save_posture_to_csv(person_id, analysis_results, file_name='../../data/processed/posture_analysis.csv'):
    """
    Saves the posture analysis results to a CSV file.
    """
    file_exists = os.path.isfile(file_name)
    with open(file_name, 'a', newline='') as csvfile:
        fieldnames = ['timestamp', 'person_id', 'shoulder_asymmetry', 'slouch_angle']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        if not file_exists:
            writer.writeheader()
        
        writer.writerow({
            'timestamp': datetime.now().isoformat(),
            'person_id': person_id,
            'shoulder_asymmetry': analysis_results['shoulder_asymmetry'],
            'slouch_angle': analysis_results['slouch_angle']
        })
    print(f"Posture data for person {person_id} saved to {file_name}.")

def process_all_posture_images(directory='.'):
    """
    Finds all posture image pairs in a directory, analyzes them, and saves the results.
    Image pairs are expected to be named like 'p11.jpg' (front) and 'p12.jpg' (side).
    """
    image_files = glob.glob(os.path.join(directory, 'p*.jpg'))
    persons = {}
    for img_path in image_files:
        match = re.match(r'.*p(\d)1\.jpg', img_path)
        if match:
            person_id = match.group(1)
            side_image_path = img_path.replace('1.jpg', '2.jpg')
            if side_image_path in image_files:
                persons[person_id] = (img_path, side_image_path)

    for person_id, (front_image, side_image) in persons.items():
        print(f"\nProcessing person ID: {person_id}")
        posture_data = analyze_posture(front_image, side_image)
        if posture_data:
            save_posture_to_csv(person_id, posture_data)

if __name__ == '__main__':
    # Process all posture images in the current directory and save the data.
    process_all_posture_images()



Processing person ID: 1
--- Analyzing Front View: .\p11.jpg ---
Shoulder Asymmetry (Y-diff): 1.99 pixels.

--- Analyzing Side View: .\p12.jpg ---
Slouch Angle (Ear-Shoulder-Hip): 179.22 degrees.
Posture data for person 1 saved to posture_analysis.csv.

Processing person ID: 2
--- Analyzing Front View: .\p21.jpg ---
Shoulder Asymmetry (Y-diff): 11.57 pixels.

--- Analyzing Side View: .\p22.jpg ---
Slouch Angle (Ear-Shoulder-Hip): 174.21 degrees.
Posture data for person 2 saved to posture_analysis.csv.

Processing person ID: 3
--- Analyzing Front View: .\p31.jpg ---
Shoulder Asymmetry (Y-diff): 0.50 pixels.

--- Analyzing Side View: .\p32.jpg ---
Slouch Angle (Ear-Shoulder-Hip): 179.73 degrees.
Posture data for person 3 saved to posture_analysis.csv.
